## AgenticVision Main Orchestrator script:

In [1]:
# Version 1.0: Last updated 29/04/2026

# Pipeline execution order:
# 1) Capture Orchestrator
# 2) Filter_pcaps
# 3) feature_extractor


# Change log:
# - Added deltas for ITAT plots, enabling 2d visualisation of LLM fingerprints wrt. ITAT
# - Added filter_pcap script to filter traffic to UDP and TCP protocols from 27/05/26

# Notes:
# - Check file output window sizes - method to find optimal window size for model evaluation??
# - Update to include additional LLM API's. Need to intergrate Gemini and others. 3 total?

# Known issues:
# -
# -


# Next Steps:
#- Add more LLM Api's
#- Identify t-SNE plot for different llm & workload types
# - Create comparative prompts where the output should be exactly the same

In [13]:
"""
capture_orchestrator.py
=======================

Orchestrates network traffic capture for LLM inference sessions and
records aligned metadata for downstream analysis.

This module:
- Launches `tshark` to capture packet-level traffic per session
- Executes prompts against configured LLM providers (e.g., Groq)
- Streams and records token-level response timing
- Saves outputs as:
    - `.pcap` files (raw network traffic)
    - `.json` metadata (session details)
    - `.tokens.json` (token-level timing traces)

Each session corresponds to a single (LLM, prompt) pair and is uniquely
identified for reproducibility and analysis.

Workflow:
    1. Start packet capture
    2. Send prompt to LLM API (streaming)
    3. Record token arrival times and full response
    4. Stop capture and persist artifacts

Outputs:
    captures/
        ├── <timestamp>_<llm>_<workload>_<id>.pcap
        ├── <timestamp>_<llm>_<workload>_<id>.json
        └── <timestamp>_<llm>_<workload>_<id>.tokens.json

Configuration:
    - LLM providers defined in `LLM_CONFIGS`
    - Prompts loaded dynamically from `prompt_bank`
    - API keys read from environment variables (.env supported)

Requirements:
    pip install groq openai anthropic httpx requests python-dotenv
    tshark installed and available on system path

    Linux:
        sudo apt install tshark

    macOS:
        brew install wireshark

Usage:
    python capture_orchestrator.py --runs 3 --delay 2

Arguments:
    --runs        Number of repetitions per (LLM, prompt) pair
    --delay       Delay (seconds) between sessions
    --llm         Optional substring filter for LLM names
    --workload    Optional filter for prompt workload type

Notes:
    - Requires sufficient privileges to capture network traffic
    - Ensure correct network interface is configured
    - Streaming responses are used to capture fine-grained latency

Limitations:
    - Token counts are approximated via whitespace splitting
    - No retry logic for failed API calls
    - Capture accuracy depends on system/network conditions
"""

#Load in libraries
import argparse
import json
import os
import signal
import subprocess
import time
import uuid
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Optional
from dotenv import load_dotenv

load_dotenv()


# ── Optional SDK imports (only imported if key is set) ────────────────────────
# Ideally 4 LLM's:

# Import OpenAI SDK (used for OpenRouter's OpenAI-compatible endpoint)
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None  # type: ignore
    
#Import Groq LLM api
try:
    from groq import Groq
except ImportError:
    Groq = None  # type: ignore

#Import Gemini LLM api
try:
    from google import genai
except ImportError:
    genai = None  # type: ignore
 
# ─────────────────────────────────────────────────────────────────────────────
# Test toggle
# ─────────────────────────────────────────────────────────────────────────────

import importlib

#Set for full bank of prompts or test bank (smaller)
test_run = False

#^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^










if test_run:
    #import test_prompt_bank as prompt_bank
    import signiture_prompts as prompt_bank
else:
    import prompt_bank

importlib.reload(prompt_bank) # To ensure using latest version of the prompt bank

PROMPT_BANK = prompt_bank.PROMPT_BANK
Prompt = prompt_bank.Prompt
 
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
#Setup data storage location
try:
    BASE_DIR = Path(__file__).resolve().parent.parent # for .py
except NameError:
    BASE_DIR = Path.cwd().parent  # fallback for notebooks

OUTPUT_DIR = BASE_DIR / "Datastore" / "raw_captures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
 
# Network interface for tshark. Use "any" on Linux, or find yours with:
#   tshark -D
CAPTURE_INTERFACE = r"\Device\NPF_{2C768E8E-C50A-4740-ABB3-2AF5ADB15BA2}"
 
# Seconds to wait between sessions to let TCP flows settle
INTER_SESSION_DELAY = 2.0
 
# Map of LLM providers. Each entry needs a callable that takes a prompt string
# and returns (response_text, latency_seconds).
# Add/remove providers here; the rest of the pipeline is automatic.

LLM_CONFIGS: Dict[str, Dict[str, Any]] = {
    
    # "openrouter_openai__gpt_oss-120b": {
    #     "api_key_env": "OPENROUTER_API_KEY",
    #     "model": "openai/gpt-oss-120b:free",
    #     "provider": "openrouter",
    # },
    
    "groq_qwen3_32b": {
        "api_key_env": "GROQ_API_KEY",
        "model": "qwen/qwen3-32b",
        "provider": "groq",
    },
    
    "groq_llama-3.3-70b-versatile": {
        "api_key_env": "GROQ_API_KEY",
        "model": "llama-3.3-70b-versatile",
        "provider": "groq",
    },
    
    "gemini_3.1_flash-lite": {
        "api_key_env" :"GEMINI_API_KEY",
        "model": "gemini-3.1-flash-lite",
        "provider": "gemini"
    },
    
    "openai/gpt-oss-120b": {
        "api_key_env" :"GROQ_API_KEY",
        "model": "openai/gpt-oss-120b",
        "provider": "groq"
    },
    
    
    
}

 
# ─────────────────────────────────────────────────────────────────────────────
# DATA MODEL
# ─────────────────────────────────────────────────────────────────────────────
 
@dataclass
class SessionMeta:
    session_id: str
    llm_name: str
    model: str
    workload: str
    prompt_text: str
    expected_tokens: str
    tags: list
    timestamp_utc: str
    pcap_path: str
    response_text: str
    response_tokens_approx: int
    latency_seconds: float
    error: Optional[str]
 
 
# ─────────────────────────────────────────────────────────────────────────────
# LLM CALLERS
# ─────────────────────────────────────────────────────────────────────────────
    
def call_openrouter_stream(model: str, api_key: str, prompt: str):
    if OpenAI is None:
        raise ImportError("openai package not installed (required for OpenRouter)")

    client = OpenAI(
        api_key=api_key,
        base_url="https://openrouter.ai/api/v1",
    )

    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
        extra_headers={
            # Optional but recommended by OpenRouter for attribution/rate-limit tracking
            "HTTP-Referer": "https://github.com/your-repo",  # replace with your repo/site
            "X-Title": "AgenticVision",
        },
    )
    return stream
 
def call_groq_stream(model: str, api_key: str, prompt: str):
    if Groq is None:
        raise ImportError("groq package not installed")

    client = Groq(api_key=api_key)

    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )

    return stream


def call_gemini_stream(model: str, api_key: str, prompt: str):
    if genai is None:
        raise ImportError("google-genai package not installed")

    client = genai.Client(api_key=api_key)

    # Consume the stream HERE while client is still in scope
    for chunk in client.models.generate_content_stream(
        model=model,
        contents=[{"role": "user", "parts": [{"text": prompt}]}],
    ):
        yield chunk

 
CALLER_MAP = {
    "openrouter": call_openrouter_stream,
    "groq": call_groq_stream,
    "gemini": call_gemini_stream,

}

def extract_token_openrouter(chunk) -> Optional[str]:
    try:
        return chunk.choices[0].delta.content or None
    except (AttributeError, IndexError):
        return None

def extract_token_groq(chunk) -> Optional[str]:
    try:
        return chunk.choices[0].delta.content or None
    except (AttributeError, IndexError):
        return None

def extract_token_gemini(chunk) -> Optional[str]:
    try:
        return chunk.text or None
    except AttributeError:
        return None

TOKEN_EXTRACTOR_MAP = {
    "openrouter": extract_token_openrouter,
    "groq": extract_token_groq,
    "gemini": extract_token_gemini,
}
 
#Every provider needs both a caller and a token extractor
assert set(CALLER_MAP.keys()) == set(TOKEN_EXTRACTOR_MAP.keys())


# ─────────────────────────────────────────────────────────────────────────────
# TSHARK HELPERS
# ─────────────────────────────────────────────────────────────────────────────
 
def start_capture(pcap_path: Path) -> subprocess.Popen:
    cmd = [
        r"C:\Program Files\Wireshark\tshark.exe",
        "-i", CAPTURE_INTERFACE,
        "-w", str(pcap_path),
        "-q",
    ]

    #print("tshark command:", " ".join(cmd))

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE
    )

    return proc
 
 
def stop_capture(proc: subprocess.Popen) -> None:
    """Gracefully stop tshark and wait for it to flush the pcap."""
    proc.send_signal(signal.SIGTERM)
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()
 
 
# ─────────────────────────────────────────────────────────────────────────────
# SESSION RUNNER
# ─────────────────────────────────────────────────────────────────────────────
 
def run_session(llm_name: str, cfg: Dict[str, Any], prompt: Prompt) -> SessionMeta:
    session_id = str(uuid.uuid4())[:8]
    ts = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
    safe_llm = llm_name.replace("/", "_")
    
    pcap_filename = f"{ts}_{safe_llm}_{prompt.workload}_{session_id}.pcap"
    pcap_path = (OUTPUT_DIR / pcap_filename).resolve()
    #print("PCAP path:", pcap_path)
 
    api_key = os.environ.get(cfg["api_key_env"], "")
    if not api_key:
        print(f"  [SKIP] {llm_name}: env var {cfg['api_key_env']} not set")
        return SessionMeta(
            session_id=session_id,
            llm_name=llm_name,
            model=cfg["model"],
            workload=prompt.workload,
            prompt_text=prompt.text,
            expected_tokens=prompt.expected_tokens,
            tags=prompt.tags,
            timestamp_utc=ts,
            pcap_path=str(pcap_path),
            response_text="",
            response_tokens_approx=0,
            latency_seconds=0.0,
            error=f"API key not set: {cfg['api_key_env']}",
        )
 
    caller = CALLER_MAP.get(cfg["provider"])
    if caller is None:
        raise ValueError(f"Unknown provider: {cfg['provider']}")
 
    print(f"  [{llm_name}] Capturing -> {pcap_filename}")
    proc = start_capture(pcap_path)
    time.sleep(2) 
 
    response_text = ""
    latency = 0.0
    error = None
    
    try:
        stream = caller(cfg["model"], api_key, prompt.text)
        token_series = []
        full_text = ""

        extractor = TOKEN_EXTRACTOR_MAP.get(cfg["provider"])
        if extractor is None:
            raise ValueError(f"No token extractor for provider: {cfg['provider']}")

        t0 = time.perf_counter()

        for chunk in stream:
            token = extractor(chunk)
            if token:
                now = time.perf_counter()
                full_text += token
                token_series.append({"t_rel": now - t0, "token": token})

        latency = time.perf_counter() - t0
        
        # Second pass: add delta_prev and delta_next
        for i, entry in enumerate(token_series):
            entry["delta_prev"] = entry["t_rel"] - token_series[i - 1]["t_rel"] if i > 0 else None
            entry["delta_next"] = token_series[i + 1]["t_rel"] - entry["t_rel"] if i < len(token_series) - 1 else None
            
            
        response_text = full_text
        
        token_path = pcap_path.with_suffix(".tokens.json")
        with open(token_path, "w") as f:
            json.dump(token_series, f, indent=2)

    except Exception as exc:
        error = str(exc)
        print(f"API error: {exc}")
        
    finally:
        time.sleep(2) 
        stop_capture(proc)
 
    meta = SessionMeta(
        session_id=session_id,
        llm_name=llm_name,
        model=cfg["model"],
        workload=prompt.workload,
        prompt_text=prompt.text,
        expected_tokens=prompt.expected_tokens,
        tags=prompt.tags,
        timestamp_utc=ts,
        pcap_path=str(pcap_path),
        response_text=response_text,
        response_tokens_approx=len(response_text.split()),
        latency_seconds=round(latency, 4),
        error=error,
    )
 
    # Save metadata alongside the pcap
    
        
    meta_path = pcap_path.with_suffix(".json")
    with open(meta_path, "w") as f:
        json.dump(asdict(meta), f, indent=2)
 
    print(f"Latency={latency:.2f}s  ~{meta.response_tokens_approx} words")
    return meta
 
 
# ─────────────────────────────────────────────────────────────────────────────
# MAIN ORCHESTRATION LOOP
# ─────────────────────────────────────────────────────────────────────────────
 
def run_all(runs: int = 1, delay: float = INTER_SESSION_DELAY,
            llm_filter: Optional[str] = None,
            workload_filter: Optional[str] = None) -> None:
    """
    Iterate over every (LLM x prompt) pair, repeated `runs` times.
    Optionally filter by llm_name or workload type.
    """
    targets = {k: v for k, v in LLM_CONFIGS.items()
               if llm_filter is None or llm_filter in k}
    prompts = [p for p in PROMPT_BANK
               if workload_filter is None or p.workload == workload_filter]
 
    total = len(targets) * len(prompts) * runs
    print(f"\nStarting capture run: {len(targets)} LLMs x {len(prompts)} prompts x {runs} run(s) = {total} sessions\n")
 
    completed = 0
    for run_idx in range(runs):
        print(f"── Run {run_idx + 1}/{runs} ─────────────────────────────────────────")
        for llm_name, cfg in targets.items():
            for prompt in prompts:
                run_session(llm_name, cfg, prompt)
                completed += 1
                print(f"  Progress: {completed}/{total}")
                time.sleep(delay)
 
    print(f"\nDone. Captures saved to: {OUTPUT_DIR.resolve()}")
 
 
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="LLM traffic capture orchestrator")
    parser.add_argument("--runs", type=int, default=1,
                        help="Number of repetitions per (LLM, prompt) pair")
    parser.add_argument("--delay", type=float, default=INTER_SESSION_DELAY,
                        help="Seconds between sessions")
    parser.add_argument("--llm", type=str, default=None,
                        help="Filter to LLM names containing this string")
    parser.add_argument("--workload", type=str, default=None,
                        help="Filter to a specific workload type")
    import sys

    if "ipykernel" in sys.modules:
        args = parser.parse_args(args=[])
    else:
        args = parser.parse_args()
 
    run_all(runs=args.runs, delay=args.delay, llm_filter=args.llm, workload_filter=args.workload)


Starting capture run: 4 LLMs x 43 prompts x 1 run(s) = 172 sessions

── Run 1/1 ─────────────────────────────────────────
  [groq_qwen3_32b] Capturing -> 20260701T152128Z_groq_qwen3_32b_short_factual_58cb5f85.pcap
Latency=0.77s  ~274 words
  Progress: 1/172
  [groq_qwen3_32b] Capturing -> 20260701T152135Z_groq_qwen3_32b_short_factual_a5636ce6.pcap
Latency=2.18s  ~481 words
  Progress: 2/172
  [groq_qwen3_32b] Capturing -> 20260701T152143Z_groq_qwen3_32b_short_factual_299bf29c.pcap
Latency=0.79s  ~260 words
  Progress: 3/172
  [groq_qwen3_32b] Capturing -> 20260701T152150Z_groq_qwen3_32b_short_factual_50500d4f.pcap
Latency=2.41s  ~566 words
  Progress: 4/172
  [groq_qwen3_32b] Capturing -> 20260701T152159Z_groq_qwen3_32b_short_factual_3a0cae3f.pcap
Latency=0.91s  ~277 words
  Progress: 5/172
  [groq_qwen3_32b] Capturing -> 20260701T152207Z_groq_qwen3_32b_short_factual_69f88b40.pcap
Latency=3.42s  ~1014 words
  Progress: 6/172
  [groq_qwen3_32b] Capturing -> 20260701T152217Z_groq_qwen3_

Latency=0.53s  ~6 words
  Progress: 87/172
  [gemini_3.1_flash-lite] Capturing -> 20260701T153457Z_gemini_3.1_flash-lite_short_factual_09de4499.pcap


Latency=0.58s  ~6 words
  Progress: 88/172
  [gemini_3.1_flash-lite] Capturing -> 20260701T153504Z_gemini_3.1_flash-lite_short_factual_850bee7a.pcap


Latency=0.91s  ~14 words
  Progress: 89/172
  [gemini_3.1_flash-lite] Capturing -> 20260701T153511Z_gemini_3.1_flash-lite_short_factual_64b49439.pcap


Latency=0.55s  ~15 words
  Progress: 90/172
  [gemini_3.1_flash-lite] Capturing -> 20260701T153517Z_gemini_3.1_flash-lite_short_factual_76095b50.pcap


...

Latency=0.96s  ~117 words
  Progress: 125/172
  [gemini_3.1_flash-lite] Capturing -> 20260701T154122Z_gemini_3.1_flash-lite_roleplay_agent_d4b4fff2.pcap


Latency=0.62s  ~39 words
  Progress: 126/172
  [gemini_3.1_flash-lite] Capturing -> 20260701T154128Z_gemini_3.1_flash-lite_instruction_following_6c4f4494.pcap


Latency=1.33s  ~35 words
  Progress: 127/172
  [gemini_3.1_flash-lite] Capturing -> 20260701T154136Z_gemini_3.1_flash-lite_instruction_following_f3e7c34d.pcap


Latency=0.69s  ~50 words
  Progress: 128/172
  [gemini_3.1_flash-lite] Capturing -> 20260701T154142Z_gemini_3.1_flash-lite_instruction_following_d6284d96.pcap


Latency=0.82s  ~51 words
  Progress: 129/172
  [openai/gpt-oss-120b] Capturing -> 20260701T154149Z_openai_gpt-oss-120b_short_factual_b28c5833.pcap
Latency=0.10s  ~6 words
  Progress: 130/172
  [openai/gpt-oss-120b] Capturing -> 20260701T154156Z_openai_gpt-oss-120b_short_factual_97ba77f4.pcap
Latency=0.08s  ~6 words
  Progress: 131/172
  [openai/gpt-oss-120b] Capturing -> 20260701T154202Z_openai_gpt-oss-120b_short_factual_a9b27004.pcap
Latency=0.17s  ~34 words
  Progress: 132/172
  [openai/gpt-oss-120b] Capturing -> 20260701T154209Z_openai_gpt-oss-120b_short_factual_e366a331.pcap
Latency=0.14s  ~14 words
  Progress: 133/172
  [openai/gpt-oss-120b] Capturing -> 20260701T154215Z_openai_gpt-oss-120b_short_factual_f5c1f9de.pcap
Latency=0.14s  ~23 words
  Progress: 134/172
  [openai/gpt-oss-120b] Capturing -> 20260701T154222Z_openai_gpt-oss-120b_short_factual_036531a5.pcap
Latency=0.08s  ~5 words
  Progress: 135/172
  [openai/gpt-oss-120b] Capturing -> 20260701T154228Z_openai_gpt-oss-120b_sh